In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# %cd /kaggle/working

# !git clone https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
# %cd /kaggle/working/WeedsGalore_Ipr28

# !ls -lah
%cd /kaggle/working
!rm -rf WeedsGalore_Ipr28
!git clone -q https://github.com/ThaoTo2806/WeedsGalore_Ipr28.git
%cd /kaggle/working/WeedsGalore_Ipr28
!git log --oneline -1
REPO = '/kaggle/working/WeedsGalore_Ipr28'

/kaggle/working
/kaggle/working/WeedsGalore_Ipr28
67367ba (HEAD -> main, origin/main, origin/HEAD) Exp16 doc


In [2]:
!pip install -q "transformers==4.45.2" torchmetrics absl-py opencv-python-headless pytest scikit-learn thop
import transformers, torchmetrics, sklearn
print('transformers', transformers.__version__, '| torchmetrics', torchmetrics.__version__, '| sklearn', sklearn.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 700.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.1 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 4.45.2 | torchmetrics 1.9.0 | sklearn 1.6.1


In [ ]:
# !pip install -q transformers==4.45.2

In [ ]:
# !pip install -q datasets huggingface_hub

In [ ]:
# from huggingface_hub import snapshot_download

# dataset_path = snapshot_download(
#     repo_id="Voxel51/weeds-galore",
#     repo_type="dataset",
#     local_dir="/kaggle/working/weeds-galore"
# )

# print(dataset_path)

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q absl-py==1.3.0 torchmetrics==0.11.4 tensorboard==2.11.0

In [ ]:
# %cd /kaggle/working/WeedsGalore_Ipr28

# !pip install -q --upgrade "tensorboard>=2.20,<2.21"

In [ ]:
# !cp -r /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset /kaggle/working/weedsgalore-dataset

In [ ]:
# !rm -rf /kaggle/working/weedsgalore-dataset

In [3]:
%cd /kaggle/working

!wget -O weedsgalore-dataset.zip \
"https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip"

/kaggle/working
--2026-11-12 09:13:19--  https://doidata.gfz.de/weedsgalore_e_celikkan_2024/weedsgalore-dataset.zip
Resolving doidata.gfz.de (doidata.gfz.de)... 139.17.228.163
Connecting to doidata.gfz.de (doidata.gfz.de)|139.17.228.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 336989804 (321M) [application/zip]
Saving to: ‘weedsgalore-dataset.zip’

weedsgalore-dataset 100%[===================>] 321.38M  17.7MB/s    in 20s     

2026-11-12 09:13:41 (16.0 MB/s) - ‘weedsgalore-dataset.zip’ saved [336989804/336989804]



In [4]:
!unzip -q /kaggle/working/weedsgalore-dataset.zip \
    -d /kaggle/working/

In [5]:
import glob, os
# 1) Ưu tiên dataset đã Add Input (thư mục chứa các ngày 2023-05-25, ...)
hits = sorted(glob.glob('/kaggle/input/**/2023-05-25/semantics', recursive=True))
if hits:
    DS = os.path.dirname(os.path.dirname(hits[0]))
elif os.path.isdir('/kaggle/working/weedsgalore-dataset/2023-05-25'):
    DS = '/kaggle/working/weedsgalore-dataset'
else:
    # 2) Nếu chưa có: tải theo cách bạn đã dùng trước đây vào /kaggle/working/weedsgalore-dataset
    #    (trang dữ liệu: https://doidata.gfz.de/weedsgalore_e_celikkan_2024/)
    raise SystemExit('Chưa tìm thấy dataset: hãy Add Input hoặc tải về /kaggle/working/weedsgalore-dataset')
print('DS =', DS)
for d in sorted(os.listdir(DS)):
    p = os.path.join(DS, d)
    if os.path.isdir(p):
        print(d, sorted(os.listdir(p)))

DS = /kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset
2023-05-25 ['images', 'instances', 'semantics']
2023-05-30 ['images', 'instances', 'semantics']
2023-06-06 ['images', 'instances', 'semantics']
2023-06-15 ['images', 'instances', 'semantics']
splits ['test.txt', 'train.txt', 'val.txt']


In [6]:
import inspect, os, sys
REPO = '/kaggle/working/WeedsGalore_Ipr28'
os.chdir(REPO)
from transformers.models.segformer.modeling_segformer import (
    SegformerEncoder, SegformerEfficientSelfAttention, SegformerOverlapPatchEmbeddings)
src_attn = inspect.getsource(SegformerEfficientSelfAttention.forward)
src_enc = inspect.getsource(SegformerEncoder.forward)
checks = {
    'Q computed before SR (query on un-reduced hidden_states)':
        src_attn.find('self.query(hidden_states)') < src_attn.find('self.sr('),
    'SR conv called on 4D tensor then layer_norm': 'self.sr(hidden_states)' in src_attn and 'self.layer_norm(hidden_states)' in src_attn,
    'encoder loop: embedding -> blocks -> norm -> reshape': all(s in src_enc for s in ['embedding_layer(hidden_states)', 'blk(hidden_states, height, width', 'norm_layer(hidden_states)']),
}
for k, v in checks.items():
    print('OK ' if v else 'FAIL', k)
os.makedirs('results/env', exist_ok=True)
with open('results/env/hf_source_check.txt', 'w') as f:
    f.write(src_attn + '\n\n' + src_enc)

import sys
if f'{REPO}/src' not in sys.path:
    sys.path.insert(0, f'{REPO}/src')
assert os.path.exists(f'{REPO}/src/nets/ccmc_sr.py'), 'Repo chưa có code mới: commit + push rồi chạy lại S2'
from nets.ccmc_sr import CCMCSegFormer
m = CCMCSegFormer(prior_mode='learned_weed')
for name, mod in m.named_modules():
    if name.endswith('.sr'):
        print(name, type(mod).__name__, tuple(mod.weight.shape), 'stride', mod.stride)
assert all(checks.values()), 'API HF khác giả định -> dừng lại, gửi file results/env/hf_source_check.txt để review'

OK  Q computed before SR (query on un-reduced hidden_states)
OK  SR conv called on 4D tensor then layer_norm
OK  encoder loop: embedding -> blocks -> norm -> reshape


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

encoder.encoder.block.0.0.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.0.1.attention.self.sr Conv2d (32, 32, 8, 8) stride (8, 8)
encoder.encoder.block.1.0.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.1.1.attention.self.sr Conv2d (64, 64, 4, 4) stride (4, 4)
encoder.encoder.block.2.0.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)
encoder.encoder.block.2.1.attention.self.sr Conv2d (160, 160, 2, 2) stride (2, 2)


In [7]:
%cd /kaggle/working/WeedsGalore_Ipr28
!mkdir -p results/env
!python -m pytest -q tests/test_ccmc_sr.py 2>&1 | tee results/env/unit_tests.log | tail -n 15

/kaggle/working
......................................................                   [100%]
54 passed in 47.15s


In [8]:
#exp16 - preflight
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

commands = [
    ["src/scripts/exp16_patch.py"],
    ["src/scripts/exp16_arms.py"],
    ["-m", "pytest", "-q", "tests/test_exp16_point.py"],
    [
        "-m", "py_compile",
        "src/train.py",
        "src/ccmc_common.py",
        "src/evaluate_ccmc.py",
    ],
    [
        "src/scripts/exp16_diagnose.py",
        f"--dataset_path={DS}",
    ],
]

for cmd in commands:
    subprocess.run(
        [sys.executable, *cmd],
        cwd=PROJECT,
        check=True,
    )

print("EXP16 PREFLIGHT PASS")

EXP16 TRAIN PATCH PASS: /kaggle/working/WeedsGalore_Ipr28/src/train_pre_exp16.py
PATCH PASS: src/ccmc_common.py backup: /kaggle/working/WeedsGalore_Ipr28/src/ccmc_common.py.pre_exp16
PATCH PASS: src/evaluate_ccmc.py backup: /kaggle/working/WeedsGalore_Ipr28/src/evaluate_ccmc.py.pre_exp16
EXP16 PATCH PASS
EXP16 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp16/exp16_arms.json
...                                                                      [100%]
3 passed in 1.68s


Traceback (most recent call last):
  File "/kaggle/working/WeedsGalore_Ipr28/src/scripts/exp16_diagnose.py", line 7, in <module>
    from datasets import WeedsGaloreDataset
ImportError: cannot import name 'WeedsGaloreDataset' from 'datasets' (/usr/local/lib/python3.13/dist-packages/datasets/__init__.py)


CalledProcessError: Command '['/usr/bin/python3', 'src/scripts/exp16_diagnose.py', '--dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset']' returned non-zero exit status 1.

In [9]:
#Cell sửa lỗi import và chạy lại diagnose
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/scripts/exp16_diagnose.py"

s = p.read_text(encoding="utf-8")

old = "from datasets import WeedsGaloreDataset"

new = """import sys
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from datasets import WeedsGaloreDataset"""

if old in s and new not in s:
    s = s.replace(old, new, 1)
    p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp16_diagnose.py",
        f"--dataset_path={DS}",
    ],
    cwd=PROJECT,
    check=True
)

print("EXP16 DIAGNOSE PASS")

{
  "split_shapes": {
    "train": {
      "n": 104,
      "shape_counts": [
        {
          "image_and_mask_shapes": "((5, 600, 600), (600, 600))",
          "n": 104
        }
      ],
      "label_pixels": [
        34950932,
        915459,
        1573609
      ]
    },
    "val": {
      "n": 26,
      "shape_counts": [
        {
          "image_and_mask_shapes": "((5, 600, 600), (600, 600))",
          "n": 26
        }
      ],
      "label_pixels": [
        8652608,
        266797,
        440595
      ]
    }
  },
  "test_opened": false,
  "baseline_error_breakdown_note": "Use existing Exp14 diagnosis JSON if available; no fake numbers."
}
SAVED: /kaggle/working/WeedsGalore_Ipr28/results/exp16/diagnostics/shape_and_error.json
EXP16 DIAGNOSE PASS


In [11]:
#exp16_Sửa lỗi AMP dtype
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/nets/exp16_point.py"

s = p.read_text(encoding="utf-8")

old = "delta.transpose(1, 2))"
new = "delta.transpose(1, 2).to(dtype=correction.dtype))"

if old in s:
    backup = p.with_suffix(".py.before_amp_fix")
    if not backup.exists():
        shutil.copy2(p, backup)

    s = s.replace(old, new, 1)
    p.write_text(s, encoding="utf-8")

assert new in s, "Không tìm thấy đoạn scatter cần sửa"

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m", "pytest", "-q",
        "tests/test_exp16_point.py",
    ],
    cwd=PROJECT,
    check=True,
)

print("EXP16 AMP DTYPE FIX PASS")

...                                                                      [100%]
3 passed in 1.68s
EXP16 AMP DTYPE FIX PASS


In [14]:
#exp16 - train seed 0
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp16_seed0.py",
        f"--dataset_path={DS}",
    ],
    cwd=PROJECT,
    check=True,
)

print("EXP16 SEED0 COMPLETED")

RUN: /usr/bin/python3 src/train.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E16_B2_SPECTRAL_POINT_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp16/E16_B2_SPECTRAL_POINT/seed0 --arm_id=E16_B2_SPECTRAL_POINT --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp16/exp16_arms.json --log_interval=50 --ckpt_interval=1000000 --num_workers=4 --resume=True --dataset_size_train=104 --in_channels=5 --num_classes=3 --ignore_index=-1 --batch_size=8 --lr=0.0003 --weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 --epochs=100 --pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --segformer=True --ccmc=False --unet=False --dlv3p_do=False --use_attention=False --backbone=b2 --head=bilinear --use_low_level_refine=True --llr_source=raw --input_scale=1.0 --lr_sche

In [15]:
from pathlib import Path
import zipfile

P = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp16_point_seed0_backup.zip")

items = [
    "results/exp16",
    "configs/exp16",
    "src/nets/exp16_point.py",
    "src/scripts/exp16_patch.py",
    "src/scripts/exp16_arms.py",
    "src/scripts/exp16_diagnose.py",
    "src/scripts/exp16_seed0.py",
    "tests/test_exp16_point.py",
    "README_EXP16.md",
    "src/train.py",
    "src/ccmc_common.py",
    "src/evaluate_ccmc.py",
    "src/train_pre_exp16.py",
    "src/ccmc_common.py.pre_exp16",
    "src/evaluate_ccmc.py.pre_exp16",
]

with zipfile.ZipFile(
    OUT, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for rel in items:
        p = P / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        files = [p] if p.is_file() else p.rglob("*")
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(P))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert (
        "results/exp16/E16_B2_SPECTRAL_POINT/"
        "seed0/final_ema.pth"
    ) in names
    assert any(n.endswith("seed0_screen.json") for n in names)

print("BACKUP PASS:", OUT)
print("Size (GB):", round(OUT.stat().st_size / 1024**3, 2))

SKIP: README_EXP16.md
BACKUP PASS: /kaggle/working/exp16_point_seed0_backup.zip
Size (GB): 0.8


In [16]:
#Cell chẩn đoán Exp16 trước khi thiết kế Exp17
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

CKPT = (
    PROJECT
    / "results/exp16/E16_B2_SPECTRAL_POINT/seed0/final_ema.pth"
)

assert CKPT.is_file(), f"Thiếu checkpoint: {CKPT}"

SCRIPT = PROJECT / "src/scripts/exp17_point_audit.py"

SCRIPT.write_text(r'''
import argparse
import csv
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import torch
import torch.nn.functional as F

from datasets import WeedsGaloreDataset
from nets.exp16_point import SpectralPointRefineSegFormer
from ccmc_common import load_checkpoint


def scores(cm):
    result = {}
    for c, name in [(1, "crop"), (2, "weed")]:
        tp = int(cm[c, c])
        fp = int(cm[:, c].sum() - tp)
        fn = int(cm[c, :].sum() - tp)

        result[name] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision_pct": round(100 * tp / max(tp + fp, 1), 3),
            "recall_pct": round(100 * tp / max(tp + fn, 1), 3),
            "IoU_pct": round(100 * tp / max(tp + fp + fn, 1), 3),
        }
    return result


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset_path", required=True)
    ap.add_argument("--ckpt", required=True)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net = SpectralPointRefineSegFormer(
        num_classes=3,
        pretrained_backbone=True,
        in_channels=5,
        use_low_level_refine=True,
        use_multiscale_fusion=True,
        llr_source="raw",
        backbone="b2",
        point_budget=4096,
    )

    net = load_checkpoint(net, args.ckpt, device).to(device).eval()

    ds = WeedsGaloreDataset(
        dataset_path=args.dataset_path,
        dataset_size=None,
        in_bands=5,
        num_classes=3,
        is_training=False,
        split="val",
        augmentation=False,
    )

    assert len(ds) == 26, "Validation split không đúng 26 ảnh"

    captured = {}

    def hook(module, inputs, output):
        captured["coarse_half"] = output.detach()

    handle = net.classifier.register_forward_hook(hook)

    cm = np.zeros((3, 3), dtype=np.int64)
    coarse_cm = np.zeros((3, 3), dtype=np.int64)

    totals = {
        "selected": 0,
        "selected_wrong_before": 0,
        "selected_wrong_after": 0,
        "selected_corrected": 0,
        "selected_damaged": 0,
        "selected_near_gt_border": 0,
        "all_wrong_before": 0,
        "all_wrong_after": 0,
        "all_pixels": 0,
        "all_near_gt_border": 0,
        "crop_weed_selected": 0,
    }

    rows = []

    try:
        with torch.inference_mode():
            for i in range(len(ds)):
                img, _, lab = ds[i]
                x = torch.from_numpy(
                    np.ascontiguousarray(img)
                ).float().unsqueeze(0).to(device)

                gt = torch.as_tensor(
                    np.asarray(lab),
                    device=device
                ).long()

                captured.clear()

                # Full forward: coarse + point refinement
                refined_logits = net(x).float()

                coarse_half = captured["coarse_half"]
                coarse_logits = F.interpolate(
                    coarse_half.float(),
                    size=gt.shape,
                    mode="bilinear",
                    align_corners=False,
                )

                prob = coarse_logits.softmax(dim=1)
                top = prob.topk(2, dim=1).values

                uncertainty = -(
                    top[:, 0] - top[:, 1]
                ).flatten(1)

                n = min(net.point_budget, gt.numel())

                # Giống lựa chọn điểm của Exp16 lúc inference
                ids = uncertainty.topk(n, dim=1).indices[0]

                coarse = coarse_logits.argmax(dim=1)[0]
                refined = refined_logits.argmax(dim=1)[0]

                valid = (gt >= 0) & (gt < 3)

                # GT border: chỉ dùng phân tích sau inference
                g = gt.clamp(0, 2)[None, None].float()
                mx = F.max_pool2d(g, 7, stride=1, padding=3)
                mn = -F.max_pool2d(-g, 7, stride=1, padding=3)
                border = (mx != mn)[0, 0] & valid

                gt_flat = gt.flatten()
                before_flat = coarse.flatten()
                after_flat = refined.flatten()
                valid_flat = valid.flatten()

                selected_valid = valid_flat[ids]
                chosen = ids[selected_valid]

                wrong_before = (coarse != gt) & valid
                wrong_after = (refined != gt) & valid

                wrong_before_sel = (
                    before_flat[chosen] != gt_flat[chosen]
                )
                wrong_after_sel = (
                    after_flat[chosen] != gt_flat[chosen]
                )

                corrected = wrong_before_sel & ~wrong_after_sel
                damaged = ~wrong_before_sel & wrong_after_sel

                totals["selected"] += chosen.numel()
                totals["selected_wrong_before"] += int(
                    wrong_before_sel.sum().item()
                )
                totals["selected_wrong_after"] += int(
                    wrong_after_sel.sum().item()
                )
                totals["selected_corrected"] += int(
                    corrected.sum().item()
                )
                totals["selected_damaged"] += int(
                    damaged.sum().item()
                )
                totals["selected_near_gt_border"] += int(
                    border.flatten()[chosen].sum().item()
                )
                totals["crop_weed_selected"] += int(
                    ((gt_flat[chosen] == 1) |
                     (gt_flat[chosen] == 2)).sum().item()
                )
                totals["all_wrong_before"] += int(
                    wrong_before.sum().item()
                )
                totals["all_wrong_after"] += int(
                    wrong_after.sum().item()
                )
                totals["all_pixels"] += int(valid.sum().item())
                totals["all_near_gt_border"] += int(border.sum().item())

                y = gt[valid].cpu().numpy()
                p = refined[valid].cpu().numpy()
                q = coarse[valid].cpu().numpy()

                cm += np.bincount(
                    3 * y + p, minlength=9
                ).reshape(3, 3)

                coarse_cm += np.bincount(
                    3 * y + q, minlength=9
                ).reshape(3, 3)

                row = {
                    "image": str(ds.img_list[i]),
                    "selected": int(chosen.numel()),
                    "selected_errors": int(wrong_before_sel.sum()),
                    "corrected": int(corrected.sum()),
                    "damaged": int(damaged.sum()),
                    "total_errors_before": int(wrong_before.sum()),
                    "total_errors_after": int(wrong_after.sum()),
                }
                rows.append(row)

                print(
                    f"{i+1}/26 | selected errors="
                    f"{row['selected_errors']} | corrected="
                    f"{row['corrected']} | damaged="
                    f"{row['damaged']}",
                    flush=True,
                )
    finally:
        handle.remove()

    t = totals
    pct = lambda a, b: round(100 * a / max(b, 1), 3)

    report = {
        "split": "val",
        "n_images": 26,
        "tta": "none",
        "checkpoint": "final_ema.pth",
        "point_budget": 4096,
        "test_opened": False,
        "gt_used_only_for_diagnostics": True,
        "coarse_crop_weed": scores(coarse_cm),
        "refined_crop_weed": scores(cm),
        "confusion_gt_rows_pred_cols": cm.tolist(),
        "point_analysis": {
            "selected_pixels": t["selected"],
            "selected_error_rate_pct": pct(
                t["selected_wrong_before"], t["selected"]
            ),
            "global_error_rate_before_pct": pct(
                t["all_wrong_before"], t["all_pixels"]
            ),
            "selected_gt_border_pct": pct(
                t["selected_near_gt_border"], t["selected"]
            ),
            "global_gt_border_pct": pct(
                t["all_near_gt_border"], t["all_pixels"]
            ),
            "selected_crop_weed_gt_pct": pct(
                t["crop_weed_selected"], t["selected"]
            ),
            "errors_corrected": t["selected_corrected"],
            "correct_predictions_damaged": t["selected_damaged"],
            "net_corrected_pixels": (
                t["selected_corrected"] - t["selected_damaged"]
            ),
            "error_count_before": t["all_wrong_before"],
            "error_count_after": t["all_wrong_after"],
        },
    }

    out = ROOT / "results/exp17_diagnostics"
    out.mkdir(parents=True, exist_ok=True)

    (out / "exp16_point_audit.json").write_text(
        json.dumps(report, indent=2)
    )

    with (out / "exp16_point_per_image.csv").open(
        "w", newline=""
    ) as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print("\n=== EXP16 POINT AUDIT ===")
    print(json.dumps(report, indent=2))
    print("\nSAVED:", out)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(SCRIPT)],
    check=True
)

subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        f"--dataset_path={DS}",
        f"--ckpt={CKPT}",
    ],
    cwd=PROJECT,
    check=True
)

print("EXP17 DIAGNOSTICS PASS")

1/26 | selected errors=2108 | corrected=638 | damaged=684
2/26 | selected errors=1855 | corrected=313 | damaged=281
3/26 | selected errors=1524 | corrected=185 | damaged=179
4/26 | selected errors=1148 | corrected=96 | damaged=53
5/26 | selected errors=1905 | corrected=324 | damaged=296
6/26 | selected errors=1469 | corrected=127 | damaged=199
7/26 | selected errors=1002 | corrected=88 | damaged=67
8/26 | selected errors=1819 | corrected=397 | damaged=316
9/26 | selected errors=1540 | corrected=92 | damaged=228
10/26 | selected errors=1102 | corrected=67 | damaged=152
11/26 | selected errors=1592 | corrected=200 | damaged=181
12/26 | selected errors=1057 | corrected=96 | damaged=52
13/26 | selected errors=2166 | corrected=508 | damaged=993
14/26 | selected errors=1880 | corrected=334 | damaged=387
15/26 | selected errors=1658 | corrected=210 | damaged=286
16/26 | selected errors=1226 | corrected=148 | damaged=82
17/26 | selected errors=1770 | corrected=199 | damaged=250
18/26 | selecte

In [9]:
#exp15-Kiểm tra và khôi phục VBB nếu cần
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/train.py"

s = p.read_text(encoding="utf-8")

print("ASCD flags:", "flags.DEFINE_float('ascd_weight'" in s)
print("VBB flags:", "flags.DEFINE_float('vbb_weight'" in s)
print("VBB import:", "from exp14_vbb import vbb_loss" in s)
print("VBB loss call:", "vbb_loss(" in s)

# Chỉ khôi phục VBB khi chưa được tích hợp
if "flags.DEFINE_float('vbb_weight'" not in s:
    assert "EXP14_VBB_TRAIN_ONLY" not in s, (
        "train.py có dấu vết patch VBB nhưng thiếu flags. "
        "Cần kiểm tra file trước khi sửa."
    )

    subprocess.run(
        [sys.executable, "src/scripts/exp14_vbb_patch.py"],
        cwd=PROJECT,
        check=True,
    )

s = p.read_text(encoding="utf-8")

assert "flags.DEFINE_float('vbb_weight'" in s
assert "from exp14_vbb import vbb_loss" in s
assert "vbb_loss(" in s

print("EXP14 VBB SOURCE READY")

ASCD flags: True
VBB flags: False
VBB import: False
VBB loss call: False
EXP14 VBB PATCH PASS, backup: /kaggle/working/WeedsGalore_Ipr28/src/train_pre_vbb.py
EXP14 VBB SOURCE READY


In [11]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

def run(*args):
    subprocess.run(
        [sys.executable, *args],
        cwd=PROJECT,
        check=True
    )

# 1. Khôi phục registry ASCD nếu thiếu
ascd = PROJECT / "configs/exp14/exp14_ascd_arms.json"

if not ascd.is_file():
    run("src/scripts/exp14_ascd_arms.py")

# 2. Khôi phục registry VBB nếu thiếu
vbb = PROJECT / "configs/exp14/exp14_vbb_arms.json"

if not vbb.is_file():
    run("src/scripts/exp14_vbb_arms.py")

assert ascd.is_file()
assert vbb.is_file()

# 3. Tạo registry Exp15
run("src/scripts/exp15_arms.py")

# 4. Kiểm thử Exp15
run("-m", "pytest", "-q", "tests/test_exp15_losses.py")
run("-m", "py_compile", "src/train.py")

# 5. Xác minh tích hợp
s = (PROJECT / "src/train.py").read_text(encoding="utf-8")

assert "flags.DEFINE_float('vbb_weight'" in s
assert "vbb_loss(" in s

print("\nEXP15 PREFLIGHT PASS")
print("ASCD registry:", ascd.is_file())
print("VBB registry:", vbb.is_file())
print(
    "EXP15 registry:",
    (PROJECT / "configs/exp15/exp15_arms.json").is_file()
)

ASCD ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_ascd_arms.json
EXP14 VBB ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_vbb_arms.json
EXP15 ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp15/exp15_arms.json
.....                                                                    [100%]
5 passed in 1.82s

EXP15 PREFLIGHT PASS
ASCD registry: True
VBB registry: True
EXP15 registry: True


In [12]:
#Exp15-A và tự chuyển sang B nếu A không đạt
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp15_seed0.py",
        f"--dataset_path={DS}",
        "--mode=auto",
    ],
    cwd=PROJECT,
    check=True
)

RUN: /usr/bin/python3 src/train.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E15_A_CW_LOVASZ_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp15/E15_A_CW_LOVASZ/seed0 --arm_id=E15_A_CW_LOVASZ --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp15/exp15_arms.json --log_interval=50 --ckpt_interval=1000000 --num_workers=4 --resume=True --dataset_size_train=104 --in_channels=5 --num_classes=3 --ignore_index=-1 --batch_size=8 --lr=0.0003 --weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 --epochs=100 --pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --segformer=True --ccmc=False --unet=False --dlv3p_do=False --use_attention=False --backbone=b2 --head=bilinear --use_low_level_refine=True --llr_source=raw --input_scale=1.0 --lr_schedule=cosine --warm

CompletedProcess(args=['/usr/bin/python3', 'src/scripts/exp15_seed0.py', '--dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset', '--mode=auto'], returncode=0)

In [13]:
#ZIP Exp15-A/B trên Kaggle
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp15_ab_backup.zip")

paths = [
    "results/exp15",
    "configs/exp15",
    "configs/exp14",
    "src/exp15_losses.py",
    "src/train.py",
    "src/train_pre_exp15.py",
    "src/scripts/exp15_patch.py",
    "src/scripts/exp15_arms.py",
    "src/scripts/exp15_seed0.py",
    "tests/test_exp15_losses.py",
]

with zipfile.ZipFile(
    OUT, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for rel in paths:
        p = PROJECT / rel

        if not p.exists():
            print("SKIP:", rel)
            continue

        files = [p] if p.is_file() else p.rglob("*")

        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()

    for arm in ("E15_A_CW_LOVASZ", "E15_B_SPEC_SPATIAL"):
        expected = (
            f"results/exp15/{arm}/seed0/final_ema.pth"
        )
        assert expected in names, f"Thiếu: {expected}"

print("BACKUP PASS:", OUT)
print("Size:", round(OUT.stat().st_size / 1024**3, 2), "GB")
print("Files:", len(names))

BACKUP PASS: /kaggle/working/exp15_ab_backup.zip
Size: 1.59 GB
Files: 40


In [9]:
#cell này để tạo lại registry ASCD trước:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

ascd_registry = PROJECT / "configs/exp14/exp14_ascd_arms.json"

if not ascd_registry.exists():
    subprocess.run(
        [sys.executable, "src/scripts/exp14_ascd_arms.py"],
        cwd=PROJECT,
        check=True,
    )

assert ascd_registry.is_file(), "Không tạo được registry ASCD"

# Tiếp tục tạo VBB registry và kiểm thử
for cmd in [
    [sys.executable, "src/scripts/exp14_vbb_arms.py"],
    [sys.executable, "-m", "pytest", "-q", "tests/test_exp14_vbb.py"],
    [sys.executable, "-m", "py_compile", "src/train.py"],
]:
    subprocess.run(cmd, cwd=PROJECT, check=True)

print("EXP14 VBB PREFLIGHT PASS")

ASCD ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_ascd_arms.json
EXP14 VBB ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_vbb_arms.json
...                                                                      [100%]
3 passed in 2.04s
EXP14 VBB PREFLIGHT PASS


In [11]:
#Thử nghiệm tiếp: Exp14-VBB (preflight)
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

# Không chạy lại exp14_vbb_patch.py
for cmd in [
    [sys.executable, "src/scripts/exp14_vbb_arms.py"],
    [sys.executable, "-m", "pytest", "-q", "tests/test_exp14_vbb.py"],
    [sys.executable, "-m", "py_compile", "src/train.py"],
]:
    subprocess.run(cmd, cwd=PROJECT, check=True)

print("EXP14 VBB PREFLIGHT PASS")

EXP14 VBB ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_vbb_arms.json
...                                                                      [100%]
3 passed in 2.02s
EXP14 VBB PREFLIGHT PASS


In [16]:
#Exp14-VBB Train và evaluate seed 0
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

for arm in ("E14_B2_AMP_REF_VBB", "E14_B2_VBB"):
    subprocess.run(
        [
            sys.executable,
            "src/scripts/exp14_vbb_seed0.py",
            f"--dataset_path={DS}",
            f"--arm={arm}",
        ],
        cwd=PROJECT,
        check=True,
    )

print("EXP14 VBB SEED0 COMPLETED")

RUN: /usr/bin/python3 src/train.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --seed=0 --run_name=E14_B2_AMP_REF_VBB_s0 --out_dir=/kaggle/working/WeedsGalore_Ipr28/results/exp14_vbb/E14_B2_AMP_REF_VBB/seed0 --arm_id=E14_B2_AMP_REF_VBB --exp12_common_config=/kaggle/working/WeedsGalore_Ipr28/configs/exp12/exp12_common_config.json --exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_vbb_arms.json --log_interval=50 --ckpt_interval=1000000 --num_workers=4 --resume=True --dataset_size_train=104 --in_channels=5 --num_classes=3 --ignore_index=-1 --batch_size=8 --lr=0.0003 --weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 --epochs=100 --pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --segformer=True --ccmc=False --unet=False --dlv3p_do=False --use_attention=False --backbone=b2 --head=bilinear --use_low_level_refine=True --llr_source=raw --input_scale=1.0 --lr_sched

In [17]:
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
OUT = Path("/kaggle/working/exp14_ascd_vbb_backup.zip")

paths = [
    "results/exp14_ascd",
    "results/exp14_vbb",
    "configs/exp14",
    "src/exp14_ascd.py",
    "src/exp14_vbb.py",
    "src/train.py",
    "src/train_pre_ascd.py",
    "src/train_pre_vbb.py",
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
    "src/scripts/exp14_seed0.py",
    "src/scripts/exp14_diagnose.py",
    "src/scripts/exp14_vbb_patch.py",
    "src/scripts/exp14_vbb_arms.py",
    "src/scripts/exp14_vbb_seed0.py",
    "tests/test_exp14_ascd.py",
    "tests/test_exp14_vbb.py",
]

with zipfile.ZipFile(
    OUT, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6, allowZip64=True
) as z:
    for rel in paths:
        p = PROJECT / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        files = [p] if p.is_file() else p.rglob("*")
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(OUT) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert any("exp14_vbb/E14_B2_VBB/seed0/final_ema.pth" in n
               for n in names)
    print("ZIP PASS:", OUT)
    print("Files:", len(names))
    print("Size:", round(OUT.stat().st_size / 1024**3, 2), "GB")

SKIP: results/exp14_ascd
SKIP: src/train_pre_ascd.py
ZIP PASS: /kaggle/working/exp14_ascd_vbb_backup.zip
Files: 45
Size: 1.59 GB


In [11]:
from pathlib import Path
import shutil
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
p = PROJECT / "src/train.py"
s = p.read_text(encoding="utf-8")

# Không ghi đè bản gốc
backup = PROJECT / "src/train_before_exp14_amp.py"
if not backup.exists():
    shutil.copy2(p, backup)

# 1. Đăng ký AMP nếu chưa có
if "flags.DEFINE_bool('amp'," not in s:
    anchor = "flags.DEFINE_integer('batch_size',"
    assert s.count(anchor) == 1
    pos = s.index("\n", s.index(anchor)) + 1
    s = (
        s[:pos]
        + "flags.DEFINE_bool('amp', False, 'CUDA AMP')\n"
        + s[pos:]
    )

# 2. Khởi tạo GradScaler
if "scaler = torch.amp.GradScaler(" not in s:
    anchor = "    # Metric\n"
    assert s.count(anchor) == 1
    s = s.replace(
        anchor,
        "    scaler = torch.amp.GradScaler('cuda', enabled=FLAGS.amp)\n\n"
        + anchor
    )

# 3. Autocast forward
if "with torch.autocast(" not in s:
    old = "            out, extras = forward_train(features, labels, binary_labels)"
    assert s.count(old) == 1
    s = s.replace(
        old,
        "            with torch.autocast("
        "device_type='cuda', dtype=torch.float16, enabled=FLAGS.amp):\n"
        "                out, extras = forward_train(features, labels, binary_labels)"
    )

# 4. Backward bằng GradScaler
if "scaler.scale(loss).backward()" not in s:
    old = "            loss.backward()"
    assert s.count(old) == 1
    s = s.replace(old, "            scaler.scale(loss).backward()")

# 5. Optimizer step bằng GradScaler
if "scaler.step(optimizer)" not in s:
    old = "            optimizer.step()"
    assert s.count(old) == 1
    s = s.replace(
        old,
        "            scaler.step(optimizer)\n"
        "            scaler.update()"
    )

# 6. Kiểm tra cấu trúc AMP
assert s.count("scaler.scale(loss).backward()") == 1
assert s.count("scaler.step(optimizer)") == 1
assert s.count("scaler.update()") == 1
assert "factor = lr_factor(" in s

compile(s, str(p), "exec")
p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

print("EXP14 AMP FIX PASS")
print("Backup:", backup)

EXP14 AMP FIX PASS
Backup: /kaggle/working/WeedsGalore_Ipr28/src/train_before_exp14_amp.py


In [12]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

for script in (
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
):
    subprocess.run(
        [sys.executable, script],
        cwd=PROJECT,
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_exp14_ascd.py"],
    cwd=PROJECT,
    check=True
)

subprocess.run(
    [sys.executable, "-m", "py_compile", "src/train.py"],
    cwd=PROJECT,
    check=True
)

print("EXP14 ASCD PREFLIGHT PASS")

ASCD PATCH PASS: /kaggle/working/WeedsGalore_Ipr28/src/train.py
BACKUP: /kaggle/working/WeedsGalore_Ipr28/src/train_pre_ascd.py
ASCD ARMS PASS: /kaggle/working/WeedsGalore_Ipr28/configs/exp14/exp14_ascd_arms.json
......                                                                   [100%]
6 passed in 1.88s
EXP14 ASCD PREFLIGHT PASS


In [14]:
#exp14 chạy seed 0:
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

DS = (
    "/kaggle/input/datasets/thothanh28/"
    "weedsgalore-dataset/weedsgalore-dataset"
)

BASELINE = Path(
    "/kaggle/input/datasets/thanhthao28/"
    "exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth"
)

assert BASELINE.is_file(), f"Không tìm thấy checkpoint: {BASELINE}"

print("Baseline:", BASELINE)
print("Checkpoint exists:", BASELINE.is_file())

subprocess.run(
    [
        sys.executable,
        "src/scripts/exp14_seed0.py",
        f"--dataset_path={DS}",
        f"--baseline_ckpt={BASELINE}",
    ],
    cwd=PROJECT,
    check=True
)

Baseline: /kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth
Checkpoint exists: True
RUN: /usr/bin/python3 src/scripts/exp14_diagnose.py --dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset --ckpt=/kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth --tta=d4 --boundary_px=3 --out=/kaggle/working/WeedsGalore_Ipr28/results/exp14_ascd/diagnostics/b2_seed0_d4.json
1/26 2023-06-15_0956
2/26 2023-06-06_0953
3/26 2023-05-30_0915
4/26 2023-05-25_0966
5/26 2023-06-06_1043
6/26 2023-05-30_1030
7/26 2023-05-25_1055
8/26 2023-06-06_1039
9/26 2023-05-30_1034
10/26 2023-05-25_1051
11/26 2023-05-30_0912
12/26 2023-05-25_0969
13/26 2023-06-15_0462
14/26 2023-06-06_0463
15/26 2023-05-30_0442
16/26 2023-05-25_0476
17/26 2023-06-06_0391
18/26 2023-05-30_0374
19/26 2023-05-25_0402
20/26 2023-06-06_0395
21/26 2023-05-30_0370
22/26 2023-05-25_0406
23/26 2023-06-06_0460
24/26 2023-05-30_0445
25/26 2023-05-25_047

CompletedProcess(args=['/usr/bin/python3', 'src/scripts/exp14_seed0.py', '--dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset', '--baseline_ckpt=/kaggle/input/datasets/thanhthao28/exp12-backup/exp12/P_B2_LLR/seed0/final_ema.pth'], returncode=0)

In [15]:
from pathlib import Path
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
ZIP = Path("/kaggle/working/exp14_ascd_backup.zip")

paths = [
    "results/exp14_ascd",
    "configs/exp14",
    "src/exp14_ascd.py",
    "src/scripts/exp14_ascd_patch.py",
    "src/scripts/exp14_ascd_arms.py",
    "src/scripts/exp14_seed0.py",
    "src/scripts/exp14_diagnose.py",
    "tests/test_exp14_ascd.py",
    "src/train.py",
    "src/exp12_config.py",
]

with zipfile.ZipFile(
    ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as z:
    for rel in paths:
        p = PROJECT / rel
        if not p.exists():
            print("SKIP:", rel)
            continue
        if p.is_file():
            z.write(p, rel)
        else:
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(PROJECT))

with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    assert any(x.endswith("final_ema.pth") for x in z.namelist())
    print("BACKUP PASS:", ZIP)
    print("Files:", len(z.namelist()))
    print("Size:", round(ZIP.stat().st_size / 1024**3, 2), "GB")

BACKUP PASS: /kaggle/working/exp14_ascd_backup.zip
Files: 26
Size: 0.8 GB


In [9]:
from pathlib import Path

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

assert PROJECT.exists(), f"Không tìm thấy project: {PROJECT}"
assert (PROJECT / "src/train.py").is_file()

print("PROJECT:", PROJECT)
print("PROJECT OK")

PROJECT: /kaggle/working/WeedsGalore_Ipr28
PROJECT OK


In [10]:
from pathlib import Path

train_file = PROJECT / "src/train.py"
text = train_file.read_text()

print("=== EXP13 IMPORTS ===")
for i, line in enumerate(text.splitlines(), 1):
    if "exp13" in line.lower() or "VCF" in line:
        print(f"{i}: {line}")

print("\n=== V2 PATCH EXPECTED IMPORT ===")
print(
    "Expected import exists:",
    "from nets.exp13 import EXP13_HEADS, VCFSegFormer\n" in text
)

print("\n=== EXP13 FILES ===")
for f in [
    "src/nets/exp13.py",
    "src/nets/exp13_v2.py",
    "src/scripts/exp13_v2_apply_patch.py",
    "src/scripts/exp13_v2_make_arms.py",
]:
    p = PROJECT / f
    print(f, p.exists())

=== EXP13 IMPORTS ===

=== V2 PATCH EXPECTED IMPORT ===
Expected import exists: False

=== EXP13 FILES ===
src/nets/exp13.py True
src/nets/exp13_v2.py True
src/scripts/exp13_v2_apply_patch.py True
src/scripts/exp13_v2_make_arms.py True


In [11]:
#Cell cài Exp13 v2 trên Kaggle
from pathlib import Path
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

def run(script):
    print(f"\n=== RUN {script} ===", flush=True)
    subprocess.run(
        [sys.executable, script],
        cwd=PROJECT,
        check=True
    )

train = PROJECT / "src/train.py"
common = PROJECT / "src/ccmc_common.py"

v1_import = "from nets.exp13 import EXP13_HEADS, VCFSegFormer"
v2_import = "from nets.exp13_v2 import VCFContextSegFormer"

# Bước 1: kiểm tra Exp13 v1
if not all(v1_import in p.read_text(encoding="utf-8")
           for p in (train, common)):
    print("Thiếu đăng ký Exp13 v1 -> áp dụng patch v1")
    run("src/scripts/exp13_apply_patch.py")
else:
    print("Exp13 v1 imports OK")

# Bước 2: kiểm tra và áp dụng Exp13 v2
if not all(v2_import in p.read_text(encoding="utf-8")
           for p in (train, common)):
    run("src/scripts/exp13_v2_apply_patch.py")
else:
    print("Exp13 v2 imports OK")

# Bước 3: sinh cấu hình v2
run("src/scripts/exp13_v2_make_arms.py")

# Bước 4: kiểm tra
assert all(
    v2_import in p.read_text(encoding="utf-8")
    for p in (train, common)
), "V2 registration FAILED"

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_exp13.py",
        "tests/test_exp13_v2.py"
    ],
    cwd=PROJECT,
    check=True
)

print("\nEXP13 V2 PREFLIGHT PASS")

Thiếu đăng ký Exp13 v1 -> áp dụng patch v1

=== RUN src/scripts/exp13_apply_patch.py ===
PATCHED: src/nets/segformer.py
PATCHED: src/ccmc_common.py
PATCHED: src/train.py
Done. Next: git diff, python src/scripts/exp13_make_arms.py, pytest tests/test_exp13.py

=== RUN src/scripts/exp13_v2_apply_patch.py ===
PATCHED src/nets/exp13.py
PATCHED src/train.py
PATCHED src/ccmc_common.py
VCF-Context registration complete. Run pytest and review git diff.

=== RUN src/scripts/exp13_v2_make_arms.py ===
CREATED /kaggle/working/WeedsGalore_Ipr28/configs/exp13/exp13_v2_arms.json
....                                                                     [100%]
4 passed in 3.45s

EXP13 V2 PREFLIGHT PASS


In [25]:
#Sửa AMP đúng cấu trúc train.py
from pathlib import Path
import subprocess
import sys
import shutil

p = PROJECT / "src/train.py"
backup = PROJECT / "src/train_before_amp.py"

s = p.read_text(encoding="utf-8")

# Chặn patch trùng
assert "flags.DEFINE_bool('amp'," not in s
assert "scaler.scale(loss).backward()" not in s

# Đảm bảo cấu trúc đúng
assert s.count("flags.DEFINE_integer('batch_size',") == 1
assert s.count("    # Metric\n") == 1
assert s.count(
    "            out, extras = forward_train(features, labels, binary_labels)"
) == 1
assert s.count("            loss.backward()") == 1
assert s.count("            optimizer.step()") == 1

shutil.copy2(p, backup)

# 1. Thêm cờ AMP
anchor = "flags.DEFINE_integer('batch_size',"
pos = s.index(anchor)
line_end = s.index("\n", pos)

s = (
    s[:line_end + 1]
    + "flags.DEFINE_bool('amp', False, 'CUDA mixed precision')\n"
    + s[line_end + 1:]
)

# 2. Khởi tạo GradScaler
s = s.replace(
    "    # Metric\n",
    "    scaler = torch.amp.GradScaler('cuda', enabled=FLAGS.amp)\n\n"
    "    # Metric\n"
)

# 3. Autocast cho model forward
old = "            out, extras = forward_train(features, labels, binary_labels)"

new = (
    "            with torch.autocast("
    "device_type='cuda', dtype=torch.float16, enabled=FLAGS.amp):\n"
    "                out, extras = forward_train(features, labels, binary_labels)"
)

s = s.replace(old, new)

# 4. Gradient scaling
s = s.replace(
    "            loss.backward()",
    "            scaler.scale(loss).backward()"
)

# 5. Giữ nguyên LR scheduler, chỉ sửa optimizer step
s = s.replace(
    "            optimizer.step()",
    "            scaler.step(optimizer)\n"
    "            scaler.update()"
)

# Kiểm tra trước khi ghi
compile(s, str(p), "exec")

assert "scaler.scale(loss).backward()" in s
assert "scaler.step(optimizer)" in s
assert "scaler.update()" in s
assert "factor = lr_factor(" in s

p.write_text(s, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "py_compile", str(p)],
    check=True
)

print("AMP PATCH PASS")
print("Backup:", backup)

AMP PATCH PASS
Backup: /kaggle/working/WeedsGalore_Ipr28/src/train_before_amp.py


In [26]:
#Kiểm tra AMP đã được chèn đúng
s = (PROJECT / "src/train.py").read_text()
lines = s.splitlines()

for i, line in enumerate(lines):
    if "scaler.scale(loss).backward()" in line:
        for j in range(i, min(i + 12, len(lines))):
            print(f"{j+1}: {lines[j]}")

565:             scaler.scale(loss).backward()
566:             factor = lr_factor(tot_iter, total_steps, warmup_steps, FLAGS.lr_schedule, FLAGS.min_lr_ratio)
567:             for group, base_lr in zip(optimizer.param_groups, base_lrs):
568:                 group['lr'] = base_lr * factor
569:             scaler.step(optimizer)
570:             scaler.update()
571:             if ema_model is not None:
572:                 ema_update(ema_model, net, ema_decay_at(tot_iter, FLAGS.ema_decay))
573: 
574:             accum_loss += loss.item()
575:             accum_iter += 1
576:             tot_iter += 1


In [27]:
#cell tạo arm P13_B2_VCFCTX_AMP8 và train seed 0. Giữ batch size = 8, bật AMP FP16, không ensemble và không ghi đè các arm cũ.
from pathlib import Path
from exp12_config import load_json, build_cli, check_run_config
import copy
import json
import subprocess
import sys

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")

# 1. Đường dẫn cấu hình
common_path = PROJECT / "configs/exp12/exp12_common_config.json"
old_arms_path = PROJECT / "configs/exp13/exp13_v2_arms.json"
new_arms_path = PROJECT / "configs/exp13/exp13_v2_amp_arms.json"

common = load_json(common_path)
arms = copy.deepcopy(load_json(old_arms_path))

# 2. Tạo arm riêng
base_arm = "P13_B2_VCFCTX"
arm = "P13_B2_VCFCTX_AMP8"
seed = 0

assert base_arm in arms["arms"]
assert common["common"]["batch_size"] == 8

assert "flags.DEFINE_bool('amp'," in (
    PROJECT / "src/train.py"
).read_text(), "Chưa patch AMP vào train.py"

arms["arms"][arm] = copy.deepcopy(arms["arms"][base_arm])
arms["arms"][arm]["flags"]["amp"] = True
arms["arms"][arm]["note"] = (
    "VCF-Context B2, AMP FP16, batch_size=8, seed0 exploratory"
)
arms["version"] = "exp13-v2-amp8-dev"

# 3. Lưu registry riêng, không ghi đè cấu hình cũ
content = json.dumps(arms, indent=2) + "\n"

if new_arms_path.exists():
    assert new_arms_path.read_text() == content, (
        f"Registry đã tồn tại nhưng khác nội dung: {new_arms_path}"
    )
else:
    new_arms_path.write_text(content)

# 4. Tạo output mới
out = PROJECT / "results/exp13_v2" / arm / f"seed{seed}"

assert not out.exists(), f"Run đã tồn tại: {out}"
out.mkdir(parents=True)

# 5. Tạo lệnh train
cmd = [sys.executable, "src/train.py"] + build_cli(
    common,
    arms,
    arm,
    seed,
    DS,
    str(out),
    f"{arm}_s{seed}",
    str(common_path),
    str(new_arms_path)
)

# 6. Kiểm tra trước khi chạy
assert "--batch_size=8" in cmd
assert "--amp=True" in cmd
assert "--head=vcfctx" in cmd
assert "--backbone=b2" in cmd

print("=== EXP13 V2 AMP TRAIN ===", flush=True)
print("Arm:", arm, flush=True)
print("Seed:", seed, flush=True)
print("Batch size: 8", flush=True)
print("AMP: True", flush=True)
print("Backbone: B2", flush=True)
print("Head: VCF-Context", flush=True)

# 7. Huấn luyện
subprocess.run(cmd, cwd=PROJECT, check=True)

# 8. Kiểm tra kết quả
assert (out / "final_ema.pth").is_file()
assert (out / "run_summary.json").is_file()

assert not check_run_config(
    out / "run_config.json",
    common,
    arms
)

print("\nTRAIN PASS:", arm, seed)

=== EXP13 V2 AMP TRAIN ===
Arm: P13_B2_VCFCTX_AMP8
Seed: 0
Batch size: 8
AMP: True
Backbone: B2
Head: VCF-Context
Exp12 configuration check passed: {'arm_id': 'P13_B2_VCFCTX_AMP8', 'common_sha256': 'c28cdcdf035c87b87c8bf25ca787739dd2b34b90868aeed889db4a98010ef2a0', 'arms_sha256': 'db50a0872c3a5ebf255426fd60c135a00be6a8e4f0fae25c475d319f3ea7e43b'}
Using cuda
Cuda current device: 0
Cuda device name: Tesla T4

MODEL STATISTICS
Total Parameters: 24,727,634
Trainable Parameters: 24,727,634
Non-trainable Parameters: 0
FLOPs (forward pass): 39,473,709,056 (~39.47G)
FLOPs per pixel: 150,580
Inference time (avg 100 runs): 45.74 ms
Throughput: 21.86 img/s

Logging to: /kaggle/working/WeedsGalore_Ipr28/results/exp13_v2/P13_B2_VCFCTX_AMP8/seed0
Epoch: 0 iter: 1, Loss: 0.71
mIoU: 32.91%
Epoch: 0, EMA Val mIoU: 30.81%
Epoch: 0, Val mIoU: 30.81% | c0=92.44 c1=0.00 c2=0.00
Saved best_val.pth (30.81%)
Epoch: 1, EMA Val mIoU: 37.36%
Epoch: 1, Val mIoU: 44.25% | c0=94.05 c1=22.09 c2=16.61
Saved best_val.

In [28]:
from exp12_config import eval_cli
import subprocess
import sys

arm = "P13_B2_VCFCTX_AMP8"
seed = 0

out = PROJECT / "results/exp13_v2" / arm / f"seed{seed}"
ckpt = out / "final_ema.pth"

arms_path = PROJECT / "configs/exp13/exp13_v2_amp_arms.json"
arms = load_json(arms_path)

bins = PROJECT / "doc/exp02_results/results/exp02/instance_bins.json"

assert ckpt.is_file()
assert bins.is_file()

cmd = [sys.executable, "src/evaluate_ccmc.py"] + eval_cli(
    arms,
    arm,
    str(ckpt),
    DS,
    "val",
    str(bins),
    str(out / "val_d4"),
    f"{arm}_s{seed}",
    str(arms_path),
    tta="d4"
)

subprocess.run(cmd, cwd=PROJECT, check=True)

{"run_name": "P13_B2_VCFCTX_AMP8_s0", "split": "val", "prior_override": "none", "prior_noise": "none", "GT_LEAKAGE": false, "instance_source": {"official": 26}, "lambdas": []}
mIoU 84.70 | bg 98.40 crop 77.99 weed 77.69 | BIoU crop 67.31 weed 57.65
  small  n= 620 recall@0.5=62.10 recall@0.3=72.58 cov>=0.5=70.16
  medium n= 567 recall@0.5=76.19 recall@0.3=79.89 cov>=0.5=94.18
  large  n= 621 recall@0.5=50.89 recall@0.3=58.94 cov>=0.5=98.87
  all    n=1808 recall@0.5=62.67 recall@0.3=70.19 cov>=0.5=87.56


CompletedProcess(args=['/usr/bin/python3', 'src/evaluate_ccmc.py', '--dataset_path=/kaggle/input/datasets/thothanh28/weedsgalore-dataset/weedsgalore-dataset', '--split=val', '--ckpt=/kaggle/working/WeedsGalore_Ipr28/results/exp13_v2/P13_B2_VCFCTX_AMP8/seed0/final_ema.pth', '--ccmc=False', '--bins_json=/kaggle/working/WeedsGalore_Ipr28/doc/exp02_results/results/exp02/instance_bins.json', '--boundary_px=3', '--run_name=P13_B2_VCFCTX_AMP8_s0', '--out_prefix=/kaggle/working/WeedsGalore_Ipr28/results/exp13_v2/P13_B2_VCFCTX_AMP8/seed0/val_d4', '--arm_id=P13_B2_VCFCTX_AMP8', '--exp12_arms=/kaggle/working/WeedsGalore_Ipr28/configs/exp13/exp13_v2_amp_arms.json', '--tta=d4', '--weed_bias=0.0', '--backbone=b2', '--head=vcfctx', '--use_low_level_refine=True', '--llr_source=raw', '--input_scale=1.0'], returncode=0)

In [30]:
from pathlib import Path
import subprocess
import zipfile

PROJECT = Path("/kaggle/working/WeedsGalore_Ipr28")
dst = Path("/kaggle/working/exp13_v2.zip")

# Các thư mục cần sao lưu
paths = [
    "results/exp13_v2",
]

for p in paths:
    assert (PROJECT / p).exists(), f"Không tồn tại: {p}"

# Nén, giữ nguyên cấu trúc thư mục
subprocess.run(
    ["zip", "-q", "-0", "-r", str(dst), *paths],
    cwd=PROJECT,
    check=True
)

# Kiểm tra ZIP
with zipfile.ZipFile(dst) as z:
    assert z.testzip() is None, "ZIP bị lỗi!"
    print("Số file:", len(z.namelist()))

print("BACKUP PASS:", dst)
print("Dung lượng:", round(dst.stat().st_size / 1024**3, 2), "GB")

Số file: 23
BACKUP PASS: /kaggle/working/exp13_v2.zip
Dung lượng: 0.92 GB


In [ ]:
#Cell E6.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp06'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True"
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'MCSDg':    f"{SEG} --use_low_level_refine=False --head=mcsd --spectral_guide=True",
    'MCSD':     f"{SEG} --use_low_level_refine=False --head=mcsd",
    'PS':       f"{SEG} --use_low_level_refine=False --head=pixelshuffle",
    'Dnone':    f"{SEG} --use_low_level_refine=True --llr_source=none",
    'LiteSent': f"{SEG} --use_low_level_refine=True --llr_source=raw",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E6.1 — Train seed 3, 4 (ưu tiên cặp xác nhận L0/MCSD-g)
ORDER = [('L0', 3), ('MCSDg', 3), ('L0', 4), ('MCSDg', 4), ('MCSD', 3), ('MCSD', 4),
         ('Dnone', 3), ('Dnone', 4), ('LiteSent', 3), ('LiteSent', 4), ('PS', 3), ('PS', 4)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E6.2 — Đánh giá validation, seed 0–4
def evaluate(model, seed, arch, extra='', tag=None):
    ck = f'runs/{model}_s{seed}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', model, seed); return
    name = f'{tag or model}_s{seed}'
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={name} --out_prefix={OUT}/{name}_val > {OUT}/logs/eval_{name}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{name}.log

for model in ('L0', 'MCSDg', 'MCSD', 'PS', 'Dnone', 'LiteSent'):
    for s in range(5):
        evaluate(model, s, EVAL_ARCH[model])
# Ablation: số vòng Sinkhorn lúc suy luận (cùng checkpoint MCSD-g)
for it in (1, 20):
    for s in range(5):
        evaluate('MCSDg', s, EVAL_ARCH['MCSDg'], f'--sinkhorn_iters={it}', tag=f'MCSDgIt{it}')

In [ ]:
#Cell E6.3 — Xác nhận (seed 3–4) và bảng 5 seed
def compare(a, b, gate='none', seeds=range(5), extra='', name=None):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = name or f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) XÁC NHẬN trên seed mới
compare('MCSDg', 'L0', 'abs', seeds=(3, 4), extra='--min_delta=0 --min_seeds_pass=2', name='CONFIRM_MCSDg_vs_L0_s34')
# 2) Bảng 5 seed
compare('MCSDg', 'L0', 'abs')
compare('MCSD', 'L0', 'abs')
compare('MCSD', 'PS', 'g4')
compare('MCSDg', 'MCSD')
compare('PS', 'L0')
compare('Dnone', 'L0')
compare('LiteSent', 'L0')
compare('MCSDg', 'Dnone')
compare('MCSDg', 'LiteSent')
# 3) Ablation số vòng Sinkhorn
compare('MCSDgIt1', 'MCSDg')
compare('MCSDgIt20', 'MCSDg')

In [ ]:
#Cell E6.4 — Bảo toàn khối lượng theo số vòng Sinkhorn (MCSD-g seed 0)
for it in (1, 5, 20):
    !python src/scripts/mass_check.py --dataset_path={DS} --ckpt=runs/MCSDg_s0/best_val.pth --ccmc=False --head=mcsd --spectral_guide=True --sinkhorn_iters={it} --out={OUT}/mass_check_MCSDg_s0_it{it}.json

In [ ]:
#Cell E6.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E5.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp05'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
SEG = "--ccmc=False --segformer=True --use_low_level_refine=False"
TRAIN_ARCH = {
    'MCSD':  f"{SEG} --head=mcsd",
    'PS':    f"{SEG} --head=pixelshuffle",
    'MCSDg': f"{SEG} --head=mcsd --spectral_guide=True",
}
EVAL_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'Dnone':    "--ccmc=False --use_low_level_refine=True --llr_source=none",
    'LiteSent': "--ccmc=False --use_low_level_refine=True --llr_source=raw",
    'MCSD':     "--ccmc=False --use_low_level_refine=False --head=mcsd",
    'PS':       "--ccmc=False --use_low_level_refine=False --head=pixelshuffle",
    'MCSDg':    "--ccmc=False --use_low_level_refine=False --head=mcsd --spectral_guide=True",
}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E5.1 — Train (theo thứ tự ưu tiên)
ORDER = [('MCSD', 0), ('PS', 0), ('MCSD', 1), ('PS', 1), ('MCSD', 2), ('PS', 2),
         ('MCSDg', 0), ('MCSDg', 1), ('MCSDg', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E5.2 — Đánh giá validation (mô hình mới + tham chiếu)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'Dnone', 'LiteSent', 'MCSD', 'PS', 'MCSDg'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E5.3 — Gate và so sánh
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('MCSD', 'L0', 'abs')       # H-MCSD
compare('MCSD', 'PS', 'g4')        # H-mech
compare('PS', 'L0')                # head sub-pixel không bảo toàn khối lượng
compare('MCSD', 'Dnone')           # H-eff
compare('MCSD', 'LiteSent')
compare('MCSDg', 'MCSD')           # H-spec
compare('MCSDg', 'L0', 'abs')

In [ ]:
#Cell E5.4 — Profile (600×600, cùng quy ước)
for label in ('MCSD', 'PS', 'MCSDg'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
%%writefile /kaggle/working/WeedsGalore_Ipr28/mass_check.py
import json, sys
import numpy as np, torch
sys.path.insert(0, 'src')
from nets.mcsd import MCSDSegFormer
from ccmc_common import load_checkpoint
from datasets import WeedsGaloreDataset
ds_path, ckpt, out = sys.argv[1], sys.argv[2], sys.argv[3]
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = load_checkpoint(MCSDSegFormer(head='mcsd'), ckpt, dev).to(dev).eval()
ds = WeedsGaloreDataset(ds_path, None, 5, 3, False, 'val', False)
errs, pred_mass, gt_mass = [], 0.0, 0.0
with torch.no_grad():
    for i in range(len(ds)):
        img, _, lab = ds[i]
        x = torch.from_numpy(np.ascontiguousarray(img)).float()[None].to(dev)
        logp, ex = net(x, return_aux=True)
        p = logp.exp()
        q = torch.softmax(ex['occ_logits'].float(), 1)
        errs.append((torch.nn.functional.avg_pool2d(p, 4) - q).abs().max().item())
        pred_mass += p[:, 2].sum().item()
        gt_mass += float((np.asarray(lab) == 2).sum())
res = {'max_row_marginal_error': max(errs), 'mean_row_marginal_error': float(np.mean(errs)),
       'pred_weed_mass_over_gt': pred_mass / gt_mass}
print(res)
json.dump(res, open(out, 'w'), indent=2)

In [ ]:
!python mass_check.py {DS} runs/MCSD_s0/best_val.pth {OUT}/mass_check_MCSD_s0.json

In [ ]:
#Cell E5.6 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E4.0 — Biến dùng chung
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp04'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
TRAIN_ARCH = {
    'L0':       "--ccmc=True --prior_mode=none --use_low_level_refine=False",
    'LiteSent': "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=raw",
    'Dnone':    "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=none",
    'DF1':      "--ccmc=False --segformer=True --use_low_level_refine=True --llr_source=f1",
}
EVAL_ARCH = {k: v.replace('--segformer=True ', '') for k, v in TRAIN_ARCH.items()}
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
#Cell E4.1 — Train
ORDER = [('Dnone', 0), ('DF1', 0), ('LiteSent', 1), ('Dnone', 1), ('DF1', 1),
         ('LiteSent', 2), ('Dnone', 2), ('DF1', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/best_val.pth') and (os.path.islink(f'runs/{run}/best_val.pth')
                                                       or os.path.exists(f'runs/{run}/run_summary.json')):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {TRAIN_ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2

In [ ]:
#Cell E4.2 — Đánh giá validation (cả L0 0–2 và LiteSent s0 lấy từ các Exp trước)
def evaluate(run, arch):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} --bins_json={BINS} --boundary_px=3 --run_name={run} --out_prefix={OUT}/{run}_val > {OUT}/logs/eval_{run}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{run}.log

for model in ('L0', 'LiteSent', 'Dnone', 'DF1'):
    for s in (0, 1, 2):
        evaluate(f'{model}_s{s}', EVAL_ARCH[model])

In [ ]:
#Cell E4.3 — So sánh (bootstrap phân tầng, 3 seed)
def compare(a, b, gate='none', seeds=(0, 1, 2)):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{a}_vs_{b}.json 2>&1 | tee {OUT}/logs/cmp_{a}_vs_{b}.log

compare('LiteSent', 'L0')      # tổng lợi ích của LLR (3 seed)
compare('Dnone', 'L0')         # chỉ tăng độ phân giải decoder
compare('LiteSent', 'Dnone')   # giá trị thêm của chi tiết từ băng thô
compare('LiteSent', 'DF1')     # băng thô so với F1
compare('DF1', 'Dnone')        # F1 có thêm gì không

In [ ]:
#Cell E4.4 — Profile (600×600, cùng quy ước)
for label in ('Dnone', 'DF1', 'LiteSent'):
    !python src/scripts/profile_model.py {EVAL_ARCH[label]} --label={label} --out={OUT}/profile_{label}.json 2>&1 | grep -E "GFLOPs|latency_ms_median|params_M"

In [ ]:
#Cell E4.5 — Gom và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if os.path.islink(f'{run_dir}/best_val.pth') or not os.path.exists(f'{run_dir}/history.jsonl'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E3.0 — Biến dùng chung, nối checkpoint cũ
import os, glob, json
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp03'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
BASE = "--ccmc=True --use_low_level_refine=False"
LAM_LEARN = "--lam_init=0.5 --lam_lr=0.01 --learn_lambda=True"
LAM_FIXED1 = "--lam_init=0.999 --learn_lambda=False"   # sigmoid(logit(0.999)) ≈ 1
ARCH = {
    'L0':    f"{BASE} --prior_mode=none",
    'C3a':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_LEARN}",
    'C2a':   f"{BASE} --prior_mode=gt_veg  --injection=ccmc_sr {LAM_LEARN}",
    'C3f':   f"{BASE} --prior_mode=gt_weed --injection=ccmc_sr {LAM_FIXED1}",
    'C3K5a': f"{BASE} --prior_mode=gt_weed --injection=pre_sr_gate {LAM_LEARN}",
}
# Cờ chỉ dùng cho train (evaluate_ccmc không có lam_*)
EVAL_ARCH = {k: ' '.join(t for t in v.split() if not t.startswith(('--lam_', '--learn_lambda')))
             for k, v in ARCH.items()}
# Nối checkpoint Exp01/Exp02 (chỉ đọc)
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print(sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E3.1 — Bin (phải trùng [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
assert json.load(open(BINS))['edges'] == [45.0, 147.0]

In [ ]:
# Cell E3.2 — Train (theo thứ tự ưu tiên)
ORDER = [('L0', 2), ('C3a', 0), ('C2a', 0), ('C3f', 0), ('C3a', 1), ('C2a', 1), ('C3a', 2), ('C2a', 2),
         ('C3K5a', 0), ('C3K5a', 1), ('C3K5a', 2), ('C3f', 1), ('C3f', 2)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|Traceback|Error" | tail -n 2
    !grep -E "lambdas" {OUT}/logs/train_{run}.log | tail -n 1

In [ ]:
#Cell E3.3 — Quỹ đạo λ (kiểm tra λ đã thực sự di chuyển)
for h in sorted(glob.glob('runs/*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    if not rows[0]['lambdas']:
        continue
    lam = [r['lambdas'] for r in rows]; best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:10s} ep0={[round(v,3) for v in lam[0]]} ep10={[round(v,3) for v in lam[min(10,len(lam)-1)]]} "
          f"ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} best(ep{best})={[round(v,3) for v in lam[best]]}")

In [ ]:
#Cell E3.4 — Đánh giá validation
def evaluate(run, arch, extra='', tag=None):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = tag or run
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "^mIoU|  small" {OUT}/logs/eval_{tag}.log

for s in (0, 1, 2):
    evaluate(f'L0_s{s}', EVAL_ARCH['L0'])
for model, seed in ORDER:
    if model != 'L0':
        evaluate(f'{model}_s{seed}', EVAL_ARCH[model])

In [ ]:
#Cell E3.5 — Cơ chế có hoạt động không? (cùng checkpoint, chỉ đổi prior lúc suy luận)
for model in ('C3a', 'C3f'):
    for s in (0, 1, 2):
        run = f'{model}_s{s}'
        evaluate(run, EVAL_ARCH[model], '--prior_override=uniform', tag=f'{run}_ovr-uniform')
for noise in ('dilate:3', 'drop:0.3', 'shift:4'):
    evaluate('C3a_s0', EVAL_ARCH['C3a'], f'--prior_noise={noise}', tag=f"C3a_s0_noise-{noise.replace(':','')}")

In [ ]:
#Cell E3.6 — So sánh (bootstrap phân tầng theo seed + ảnh) và A/A test
def compare(a, b, gate, seeds=(0, 1, 2), suffix_a='', suffix_b='', name=None, extra=''):
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, b); return
    fa = ','.join(f'{OUT}/{a}_s{s}{suffix_a}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}{suffix_b}_val_per_image.csv' for s in common)
    name = name or f'{a}{suffix_a}_vs_{b}{suffix_b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a}{suffix_a} --name_b={b}{suffix_b} --gate={gate} {extra} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

# 1) Cơ chế: prior GT so với prior đồng đều trên CÙNG checkpoint
compare('C3a', 'C3a', 'none', suffix_b='_ovr-uniform', name='C3a_activity')
compare('C3f', 'C3f', 'none', suffix_b='_ovr-uniform', name='C3f_activity')
# 2) G2-screen
compare('C3a', 'C2a', 'screen')
# 3) Trần mô tả và đối chứng cùng leakage
compare('C3a', 'L0', 'none'); compare('C3f', 'L0', 'none'); compare('C2a', 'L0', 'none')
compare('C3a', 'C3K5a', 'g4')
# 4) A/A test: cùng mô hình L0, khác seed, chỉ bootstrap theo ảnh
!python src/scripts/compare_runs.py --a={OUT}/L0_s0_val_per_image.csv --b={OUT}/L0_s1_val_per_image.csv --name_a=L0_s0 --name_b=L0_s1 --seed_bootstrap=False --out={OUT}/cmp_AA_L0s0_vs_L0s1.json 2>&1 | tee {OUT}/logs/cmp_AA.log

In [ ]:
#Cell E3.7 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/history.jsonl') or os.path.islink(f'{run_dir}/best_val.pth'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv > {OUT}/logs/collect.log 2>&1
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
new_ckpts = ' '.join(p for p in sorted(glob.glob('runs/*/best_val.pth')) if not os.path.islink(p))
!zip -qr /kaggle/working/{EXP}_ckpts.zip {new_ckpts}
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E2.0 — Biến dùng chung và khôi phục checkpoint Exp01
import os, glob, json, zipfile
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp02'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
ARCH = {
    'C2':   "--ccmc=True --prior_mode=gt_veg  --injection=ccmc_sr      --use_low_level_refine=False",
    'C3':   "--ccmc=True --prior_mode=gt_weed --injection=ccmc_sr      --use_low_level_refine=False",
    'C3K5': "--ccmc=True --prior_mode=gt_weed --injection=pre_sr_gate  --use_low_level_refine=False",
    'C3K6': "--ccmc=True --prior_mode=gt_weed --injection=post_sr_gate --use_low_level_refine=False",
}
# Exp01 checkpoints (zip hoặc thư mục đã giải nén trong /kaggle/input)
for z in glob.glob('/kaggle/input/**/exp01_ckpts.zip', recursive=True):
    zipfile.ZipFile(z).extractall('.')
for p in glob.glob('/kaggle/input/**/runs/*/best_val.pth', recursive=True):
    dst = os.path.join('runs', os.path.basename(os.path.dirname(p)))
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(f'{dst}/best_val.pth'):
        os.symlink(p, f'{dst}/best_val.pth')
print('Exp01 ckpts:', sorted(glob.glob('runs/*/best_val.pth')))

In [ ]:
#Cell E2.1 — Bin kích thước (phải trùng Exp01: edges = [45, 147])
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} > {OUT}/logs/bins.log 2>&1
edges = json.load(open(BINS))['edges']
print('edges', edges)
assert edges == [45.0, 147.0], 'Bin khác Exp01 -> dừng lại và gửi logs/bins.log'
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E2.2 — Train C2, C3, C3-K5, C3-K6 (seed 0 rồi seed 1)
ORDER = [('C3', 0), ('C2', 0), ('C3K5', 0), ('C3', 1), ('C2', 1), ('C3K6', 0), ('C3K5', 1), ('C3K6', 1)]
for model, seed in ORDER:
    run = f'{model}_s{seed}'
    if os.path.exists(f'runs/{run}/run_summary.json'):
        print('skip (done)', run); continue
    print('=====', run)
    !python src/train.py {COMMON} {ARCH[model]} --seed={seed} --run_name={model}__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Saved best|lambdas|Traceback|Error" | tail -n 3

In [ ]:
#Cell E2.3 — Đánh giá validation (Boundary IoU 3 px)
def evaluate(run, arch, extra=''):
    ck = f'runs/{run}/best_val.pth'
    if not os.path.exists(ck):
        print('NOT RUN', run); return
    tag = run + (extra.replace('--prior_noise=', '_noise-').replace(':', '') if extra else '')
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt={ck} {arch} {extra} --bins_json={BINS} --boundary_px=3 --run_name={tag} --out_prefix={OUT}/{tag}_val > {OUT}/logs/eval_{tag}.log 2>&1
    !grep -E "mIoU|small|GT prior" {OUT}/logs/eval_{tag}.log

# L0 / LiteSent đánh giá lại (BIoU 3 px), từ checkpoint Exp01
for seed in (0, 1):
    evaluate(f'L0_s{seed}', L0_ARCH)
evaluate('LiteSent_s0', '--ccmc=False --use_low_level_refine=True')
# Oracle models (GT leakage)
for model, seed in ORDER:
    evaluate(f'{model}_s{seed}', ARCH[model])

In [ ]:
#Cell E2.4 — Độ nhạy với prior GT bị nhiễu (C3 seed 0, cùng checkpoint)
for noise in ('dilate:3', 'erode:1', 'drop:0.3', 'shift:4'):
    evaluate('C3_s0', ARCH['C3'], f'--prior_noise={noise}')

In [ ]:
# Cell E2.4b — λ cuối và quỹ đạo λ của các mô hình oracle
import json, glob
for h in sorted(glob.glob('runs/C*/history.jsonl')):
    rows = [json.loads(l) for l in open(h)]
    lam = [r['lambdas'] for r in rows]
    best = rows[-1]['best_epoch']
    print(f"{h.split('/')[1]:9s} λ@ep0={[round(v,3) for v in lam[0]]} "
          f"λ@ep50={[round(v,3) for v in lam[min(50,len(lam)-1)]]} "
          f"λ@best(ep{best})={[round(v,3) for v in lam[best]]} λ@last={[round(v,3) for v in lam[-1]]}")

In [ ]:
#Cell E2.5 — So sánh paired và gate
def compare(a, b, gate, seeds=(0, 1)):
    # chỉ dùng các seed có ở CẢ HAI mô hình (paired)
    common = [s for s in seeds if os.path.exists(f'{OUT}/{a}_s{s}_val_per_image.csv')
              and os.path.exists(f'{OUT}/{b}_s{s}_val_per_image.csv')]
    if not common:
        print('NOT RUN', a, 'vs', b); return
    fa = ','.join(f'{OUT}/{a}_s{s}_val_per_image.csv' for s in common)
    fb = ','.join(f'{OUT}/{b}_s{s}_val_per_image.csv' for s in common)
    name = f'{a}_vs_{b}'
    !python src/scripts/compare_runs.py --a={fa} --b={fb} --name_a={a} --name_b={b} --gate={gate} --out={OUT}/cmp_{name}.json 2>&1 | tee {OUT}/logs/cmp_{name}.log

compare('C3', 'C2', 'screen')      # G2-screen (một chiều)
compare('C3', 'C3K5', 'g4')        # cùng leakage: CCMC-SR so với gate trước SR
compare('C3', 'C3K6', 'g4')        # cùng leakage: CCMC-SR so với gate sau SR
compare('C3', 'L0', 'none')        # biên độ trần, chỉ mô tả
compare('C2', 'L0', 'none')

In [ ]:
#Cell E2.6 — Gom kết quả và đóng gói
import shutil
for run_dir in sorted(glob.glob('runs/*')):
    if not os.path.exists(f'{run_dir}/run_config.json'):
        continue
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!zip -qr /kaggle/working/{EXP}_results.zip {OUT}
!zip -qr /kaggle/working/{EXP}_ckpts.zip runs/C*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
#Cell E1.0 — Biến dùng chung
import os
os.chdir('/kaggle/working/WeedsGalore_Ipr28')
EXP = 'exp01'
OUT = f'results/{EXP}'
BINS = 'results/instance_bins.json'   # bin cố định cho MỌI thí nghiệm sau này
os.makedirs(f'{OUT}/logs', exist_ok=True)
COMMON = (f"--dataset_path={DS} --dataset_size_train=104 --in_channels=5 --num_classes=3 "
          "--batch_size=8 --num_workers=4 --lr=0.0003 "
          "--weight_bg=0.9 --weight_crop=1.0 --weight_weed=1.3 --use_dice_loss=True --dice_weight=0.3 "
          "--epochs=100 --log_interval=50 --ckpt_interval=1000000 "
          "--pretrained_backbone=True --spectral_guided=False --use_multiscale_fusion=True --resume=True")
L0_ARCH = "--ccmc=True --prior_mode=none --use_low_level_refine=False"
LITE_ARCH = "--ccmc=False --segformer=True --use_low_level_refine=True"
print(COMMON)

In [ ]:
#Cell E1.1 — Bin kích thước cỏ (tam phân vị của train, khóa trước khi xem mô hình)
!python src/scripts/make_instance_bins.py --dataset_path={DS} --out={BINS} 2>&1 | tee {OUT}/logs/bins.log | head -n 60
!cp {BINS} {OUT}/instance_bins.json

In [ ]:
#Cell E1.2 — Train L0 seed 0 và seed 1
for seed in (0, 1):
    run = f'L0_s{seed}'
    !python src/train.py {COMMON} {L0_ARCH} --seed={seed} --run_name=L0__s{seed} --out_dir=runs/{run} 2>&1 | tee {OUT}/logs/train_{run}.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.3 — Train LiteSent seed 0 (cùng pipeline, để làm Diag-D0 trên checkpoint có LLR)
!python src/train.py {COMMON} {LITE_ARCH} --seed=0 --run_name=LiteSent__s0 --out_dir=runs/LiteSent_s0 2>&1 | tee {OUT}/logs/train_LiteSent_s0.log | grep --line-buffered -E "Val mIoU|Saved best|Traceback|Error|FLOPs|Total Parameters"

In [ ]:
#Cell E1.4 — Diag-D0: đánh giá validation (IoU, Boundary IoU, recall theo bin, theo ngày)
for seed in (0, 1):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --run_name=L0__s{seed} --out_prefix={OUT}/L0_s{seed}_val 2>&1 | tee {OUT}/logs/eval_L0_s{seed}.log
if os.path.exists('runs/LiteSent_s0/best_val.pth'):
    !python src/evaluate_ccmc.py --dataset_path={DS} --split=val --ckpt=runs/LiteSent_s0/best_val.pth --ccmc=False --use_low_level_refine=True --bins_json={BINS} --run_name=LiteSent__s0 --out_prefix={OUT}/LiteSent_s0_val 2>&1 | tee {OUT}/logs/eval_LiteSent_s0.log

In [ ]:
#Cell E1.5 — Diag-D0: recall cỏ nhỏ so với cỏ lớn trong cùng mô hình (bootstrap theo ảnh)
import csv, json, numpy as np
def small_vs_large(path, n_boot=10000, seed=0):
    rows = list(csv.DictReader(open(path)))
    g = lambda k: np.array([float(r[k]) for r in rows])
    out = {}
    for t in ('50', '30'):
        tp_s, n_s, tp_l, n_l = g(f'tp{t}_small'), g('n_small'), g(f'tp{t}_large'), g('n_large')
        point = 100 * (tp_l.sum() / n_l.sum() - tp_s.sum() / n_s.sum())
        rng = np.random.default_rng(seed)
        W = rng.multinomial(len(rows), np.full(len(rows), 1 / len(rows)), size=n_boot)
        with np.errstate(invalid='ignore', divide='ignore'):
            boot = 100 * ((W @ tp_l) / (W @ n_l) - (W @ tp_s) / (W @ n_s))
        boot = boot[np.isfinite(boot)]
        out[f'recall{t}_large_minus_small'] = {'delta': point, 'ci95': np.percentile(boot, [2.5, 97.5]).tolist()}
    return out
d0 = {}
for name in ['L0_s0', 'L0_s1', 'LiteSent_s0']:
    p = f'{OUT}/{name}_val_per_image.csv'
    if os.path.exists(p):
        d0[name] = small_vs_large(p)
print(json.dumps(d0, indent=2))
json.dump(d0, open(f'{OUT}/d0_small_vs_large.json', 'w'), indent=2)

In [ ]:
#Cell E1.6 — LiteSent so với L0 (paired bootstrap, seed 0, mang tính bổ trợ)
if os.path.exists(f'{OUT}/LiteSent_s0_val_per_image.csv'):
    !python src/scripts/compare_runs.py --a={OUT}/LiteSent_s0_val_per_image.csv --b={OUT}/L0_s0_val_per_image.csv --name_a=LiteSent --name_b=L0 --out={OUT}/cmp_LiteSent_vs_L0_s0.json 2>&1 | tee {OUT}/logs/cmp_LiteSent_vs_L0.log

In [ ]:
#Cell E1.7 — Diag-D2: linear probe trước/sau SR (stage 1–3)
for seed in (0, 1):
    !python src/scripts/probe_sr.py --dataset_path={DS} --ckpt=runs/L0_s{seed}/best_val.pth {L0_ARCH} --bins_json={BINS} --out_prefix={OUT}/d2_probe_L0_s{seed} 2>&1 | tee {OUT}/logs/probe_L0_s{seed}.log | tail -n 60

In [ ]:
#Cell E1.8 — Profile (cùng quy ước: 1×5×600×600, fp32, batch 1, T4)
profiles = {
    'L0': L0_ARCH,
    'LiteSent': LITE_ARCH.replace('--segformer=True ', ''),
    'L3': '--ccmc=True --prior_mode=learned_weed --injection=ccmc_sr --use_low_level_refine=False',
    'K5': '--ccmc=True --prior_mode=learned_weed --injection=pre_sr_gate --use_low_level_refine=False',
    'K6': '--ccmc=True --prior_mode=learned_weed --injection=post_sr_gate --use_low_level_refine=False',
}
for label, arch in profiles.items():
    !python src/scripts/profile_model.py {arch} --label={label} --out={OUT}/profile_{label}.json 2>&1 | tail -n 3

In [ ]:
#Cell E1.9 — Gom kết quả và đóng gói để gửi review
import shutil, glob
for run_dir in sorted(glob.glob('runs/*')):
    dst = f'{OUT}/runs_meta/{os.path.basename(run_dir)}'
    os.makedirs(dst, exist_ok=True)
    for f in ('history.jsonl', 'run_config.json', 'run_summary.json'):
        if os.path.exists(f'{run_dir}/{f}'):
            shutil.copy(f'{run_dir}/{f}', dst)
shutil.copytree('results/env', f'{OUT}/env', dirs_exist_ok=True)
!python src/scripts/collect_results.py --results_dir={OUT} --runs_dir=runs --out={OUT}/metrics.csv
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_results.zip {OUT} {BINS}
!cd /kaggle/working/WeedsGalore_Ipr28 && zip -qr /kaggle/working/{EXP}_ckpts.zip runs/*/best_val.pth
!ls -lh /kaggle/working/*.zip

In [ ]:
!pip install thop

In [ ]:
#Cell chạy E1/E2:
!python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --dataset_size_train=104 \
    --segformer=True \
    --spectral_guided=False \
    --use_spectral_indices=False \
    --use_attention=False \
    --dlv3p_do=False \
    --in_channels=5 \
    --num_classes=3 \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --batch_size=8 \
    --num_workers=4 \
    --lr=0.0003 \
    --weight_bg=0.9 \
    --weight_crop=1.0 \
    --weight_weed=1.3 \
    --use_dice_loss=True \
    --dice_weight=0.3 \
    --epochs=100 \
    --seed=0 \
    --out_dir=/kaggle/working/out_dir_segformer_no_refine_seed0 \
    --log_interval=25 \
    --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_segformer_no_refine_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=5 \
    --num_classes=3 \
    --temperature=1.0

In [ ]:
#Cell E5, RGB-only nhiều seed:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=3 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_rgb_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed0/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed1/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed2/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed3/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=val \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
!python /kaggle/working/WeedsGalore_Ipr28/src/evaluate.py \
    --dataset_path=/kaggle/working/weedsgalore-dataset \
    --split=test \
    --ckpt=/kaggle/working/out_dir_rgb_seed4/best_val.pth \
    --segformer=True \
    --pretrained_backbone=True \
    --use_low_level_refine=False \
    --spectral_guided=False \
    --in_channels=3 \
    --num_classes=3 \
    --use_low_level_refine=True \
    --temperature=1.0

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    !python /kaggle/working/WeedsGalore_Ipr28/src/train.py \
        --dataset_path=/kaggle/working/weedsgalore-dataset \
        --dataset_size_train=104 \
        --segformer=True \
        --spectral_guided=False \
        --use_spectral_indices=False \
        --use_attention=False \
        --dlv3p_do=False \
        --in_channels=5 \
        --num_classes=3 \
        --pretrained_backbone=True \
        --use_low_level_refine=True \
        --batch_size=8 \
        --num_workers=4 \
        --lr=0.0003 \
        --weight_bg=0.9 \
        --weight_crop=1.0 \
        --weight_weed=1.3 \
        --use_dice_loss=True \
        --dice_weight=0.3 \
        --epochs=100 \
        --seed={seed} \
        --out_dir=/kaggle/working/out_dir_5band_seed{seed} \
        --log_interval=25 \
        --ckpt_interval=100

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/WeedsGalore_Ipr28/src/evaluate.py")
text = file.read_text()

old = 'print("Architecture: 5-band SegFormer")'
new = '''print(f"Architecture: SegFormer ({FLAGS.in_channels}-channel, "
              f"LowLevelRefine={FLAGS.use_low_level_refine})")'''

assert old in text, "Không tìm thấy dòng cũ hoặc file đã thay đổi!"

file.write_text(text.replace(old, new, 1))
print("Đã cập nhật thành công dòng 65!")

In [ ]:
%cd /kaggle/working/WeedsGalore_Ipr28

import subprocess
import sys

cmd = [
    sys.executable,
    "src/calibrate_temperature.py",
    "--dataset_path", "/kaggle/working/weedsgalore-dataset",
    "--ckpt", "/kaggle/working/out_dir_rgb_seed0/best_val.pth",
    "--segformer=True",
    "--pretrained_backbone=True",
    "--spectral_guided=False",
    "--in_channels", "3",
    "--num_classes", "3",
    "--use_low_level_refine=True",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("Exit code:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)
print("\n===== STDERR =====")
print(result.stderr)

In [ ]:
import subprocess
import sys
import re
import numpy as np
from pathlib import Path

REPO = "/kaggle/working/WeedsGalore_Ipr28"
DATASET = "/kaggle/working/weedsgalore-dataset"

# Trong mọi subprocess, chạy từ thư mục repository
def run_command(command):
    result = subprocess.run(
        command,
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print("STDOUT:\n", result.stdout)
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.stdout + result.stderr

experiments = {
    "RGB": [
        f"/kaggle/working/out_dir_rgb_seed{i}/best_val.pth"
        for i in range(5)
    ],
    "5-band": [
        f"/kaggle/working/out_dir_5band_seed{i}/best_val.pth"
        for i in range(5)
    ],
}

results = {}

for name, checkpoints in experiments.items():
    miou_values = []
    ece_values = []
    temperatures = []

    in_channels = 3 if name == "RGB" else 5

    for seed, checkpoint in enumerate(checkpoints):
        if not Path(checkpoint).exists():
            print(f"SKIP: {checkpoint}")
            continue

        common = [
            f"--dataset_path={DATASET}",
            f"--ckpt={checkpoint}",
            "--segformer=True",
            "--pretrained_backbone=True",
            "--spectral_guided=False",
            f"--in_channels={in_channels}",
            "--num_classes=3",
            "--use_low_level_refine=True",
        ]

        calibration_log = run_command(
            [sys.executable, "src/calibrate_temperature.py"] + common
        )

        match = re.search(
            r"Fitted temperature:\s*T\s*=\s*([0-9.]+)",
            calibration_log,
        )
        if match is None:
            raise RuntimeError(
                f"Không đọc được temperature cho {name}, seed {seed}"
            )

        temperature = float(match.group(1))
        temperatures.append(temperature)

        evaluation_log = run_command([
            sys.executable,
            "src/evaluate.py",
            "--split=test",
            *common,
            f"--temperature={temperature}",
        ])

        print(evaluation_log)

        miou_match = re.search(r"mIoU:\s*([0-9.]+)%", evaluation_log)
        ece_match = re.search(r"ECE:\s*(?:tensor\()?([0-9.]+)", evaluation_log)

        if miou_match is None or ece_match is None:
            raise RuntimeError(f"Không đọc được kết quả cho {name}, seed {seed}")

        miou_values.append(float(miou_match.group(1)))
        ece_values.append(float(ece_match.group(1)))

    results[name] = {
        "mIoU": miou_values,
        "ECE": ece_values,
        "T": temperatures,
    }

print("\n===== SUMMARY =====")
for name, values in results.items():
    print(f"\n{name}")
    print("Temperatures:", values["T"])
    print(
        f"Test mIoU: {np.mean(values['mIoU']):.2f} "
        f"+/- {np.std(values['mIoU'], ddof=1):.2f}%"
    )
    print(
        f"Test ECE: {np.mean(values['ECE']):.5f} "
        f"+/- {np.std(values['ECE'], ddof=1):.5f}"
    )